# Objectif : Regression Logistique sur la proba de faire defaut (Y=1)

Premierement on va trouver pour chaque variable les modalités qui ont le tx de defaut le + élevé pour le mettre en valeur de reference

In [27]:
import numpy as np

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [28]:
df_discretise = pd.read_csv(filepath_or_buffer="../data/output/df_discretise.csv",
                 sep = ",")

In [29]:
for col in df_discretise.columns:
    print(f"======= Variable : {col} ===============\n")
    print(df_discretise[col].value_counts())
    print("\n===============================================\n")


======= Variable : Unnamed: 0.1 ===============

Unnamed: 0.1
0        1
1        1
2        1
3        1
4        1
        ..
29726    1
29727    1
29728    1
29729    1
29730    1
Name: count, Length: 29731, dtype: int64


======= Variable : Unnamed: 0 ===============

Unnamed: 0
1        1
2        1
3        1
4        1
5        1
        ..
32575    1
32576    1
32577    1
32578    1
32580    1
Name: count, Length: 29731, dtype: int64


======= Variable : person_age ===============

person_age
23    3649
22    3449
24    3289
25    2769
26    2259
27    1962
28    1673
29    1511
21    1148
30    1146
31    1023
32     858
33     765
34     621
35     556
36     477
37     428
38     336
39     267
40     227
41     211
42     162
43     139
44     125
45     100
46      84
47      81
48      70
50      46
49      45
51      34
52      33
53      27
54      22
55      20
58      16
57      14
60      13
20      12
56      11
65       8
66       8
62       7
64       7
61       6

In [30]:
def reorder_reference_by_risk(df, var, target="loan_status"):
    """
    Définit comme référence la modalité ayant le plus haut taux de défaut.
    """
    # Calcul du taux de défaut par modalité
    bad_rate = df.groupby(var)[target].mean().sort_values(ascending=False)
    ref = bad_rate.index[0]  # modalité à risque max
    print(f"→ Modalité de référence pour {var} : '{ref}' (taux de défaut = {bad_rate.iloc[0]:.3f})")

    # Reordonne les catégories
    df[var] = pd.Categorical(df[var], categories=bad_rate.index, ordered=True)
    return df, ref


In [31]:
categorical_vars = [
    "person_home_ownership",
    "loan_intent",
    "cb_person_default_on_file",
    "person_age_bin",
    "person_income_bin",
    "person_emp_length_bin",
    "loan_amnt_bin",
    "loan_percent_income_bin",
    "cb_person_cred_hist_length_bin"
]

df_encoded = df_discretise.copy()
ref_dict = {}

for var in categorical_vars:
    df_encoded, ref = reorder_reference_by_risk(df_encoded, var, target="loan_status")
    ref_dict[var] = ref

# Encodage final
X_encoded = pd.get_dummies(df_encoded[categorical_vars], drop_first=True)


→ Modalité de référence pour person_home_ownership : 'OTHER' (taux de défaut = 0.327)
→ Modalité de référence pour loan_intent : 'DEBTCONSOLIDATION' (taux de défaut = 0.290)
→ Modalité de référence pour cb_person_default_on_file : 'Y' (taux de défaut = 0.385)
→ Modalité de référence pour person_age_bin : 'person_age_Bin1' (taux de défaut = 0.259)
→ Modalité de référence pour person_income_bin : 'person_income_Bin1' (taux de défaut = 0.457)
→ Modalité de référence pour person_emp_length_bin : 'person_emp_length_Bin1' (taux de défaut = 0.287)
→ Modalité de référence pour loan_amnt_bin : 'loan_amnt_Bin5' (taux de défaut = 0.394)
→ Modalité de référence pour loan_percent_income_bin : 'loan_percent_income_Bin6' (taux de défaut = 0.711)
→ Modalité de référence pour cb_person_cred_hist_length_bin : 'cb_person_cred_hist_length_Bin1' (taux de défaut = 0.231)


In [32]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# --- 1️⃣ Définir les variables catégorielles ---
categorical_vars = [
    "person_home_ownership",
    "loan_intent",
    "cb_person_default_on_file",
    "person_age_bin",
    "person_income_bin",
    "person_emp_length_bin",
    "loan_amnt_bin",
    "loan_percent_income_bin",
    "cb_person_cred_hist_length_bin"
]

target = "loan_status"

# --- 2️⃣ Fonction pour réordonner les catégories selon le risque ---
def reorder_reference_by_risk(df, var, target="loan_status"):
    """
    Réordonne les modalités d'une variable catégorielle en mettant en premier
    celle ayant le plus fort taux de défaut (c'est elle qui deviendra la référence).
    """
    taux_defaut = df.groupby(var)[target].mean().sort_values(ascending=False)
    ref = taux_defaut.index[0]
    print(f"→ Modalité de référence pour {var} : '{ref}' (taux de défaut = {taux_defaut.iloc[0]:.3f})")

    df[var] = pd.Categorical(df[var], categories=taux_defaut.index, ordered=True)
    return df, ref

# --- 3️⃣ Application sur toutes les variables ---
df_encoded = df_discretise.copy()
ref_dict = {}

for var in categorical_vars:
    df_encoded, ref = reorder_reference_by_risk(df_encoded, var, target=target)
    ref_dict[var] = ref

# --- 4️⃣ Encodage en variables indicatrices (One-Hot Encoding) ---
X = pd.get_dummies(df_encoded[categorical_vars], drop_first=True)

# --- 5️⃣ Ajout de la constante ---
X = sm.add_constant(X)

# --- 6️⃣ Conversion en float ---
X = X.astype(float)

# --- 7️⃣ Cible ---
y = df_encoded[target]

# --- 8️⃣ Régression logistique ---
logit_model = sm.Logit(y, X)
result = logit_model.fit(disp=1)

# --- 9️⃣ Résumé du modèle ---
print(result.summary())

# --- 🔟 Récapitulatif des références utilisées ---
print("\n=== Modalités de référence ===")
for var, ref in ref_dict.items():
    print(f"{var} → {ref}")


→ Modalité de référence pour person_home_ownership : 'OTHER' (taux de défaut = 0.327)
→ Modalité de référence pour loan_intent : 'DEBTCONSOLIDATION' (taux de défaut = 0.290)
→ Modalité de référence pour cb_person_default_on_file : 'Y' (taux de défaut = 0.385)
→ Modalité de référence pour person_age_bin : 'person_age_Bin1' (taux de défaut = 0.259)
→ Modalité de référence pour person_income_bin : 'person_income_Bin1' (taux de défaut = 0.457)
→ Modalité de référence pour person_emp_length_bin : 'person_emp_length_Bin1' (taux de défaut = 0.287)
→ Modalité de référence pour loan_amnt_bin : 'loan_amnt_Bin5' (taux de défaut = 0.394)
→ Modalité de référence pour loan_percent_income_bin : 'loan_percent_income_Bin6' (taux de défaut = 0.711)
→ Modalité de référence pour cb_person_cred_hist_length_bin : 'cb_person_cred_hist_length_Bin1' (taux de défaut = 0.231)
Optimization terminated successfully.
         Current function value: 0.387876
         Iterations 7
                           Logit Reg

In [33]:
df_discretise.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'person_age', 'person_income',
       'person_home_ownership', 'person_emp_length', 'loan_intent',
       'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_status',
       'loan_percent_income', 'cb_person_default_on_file',
       'cb_person_cred_hist_length', 'person_age_bin', 'person_income_bin',
       'person_emp_length_bin', 'loan_amnt_bin', 'loan_percent_income_bin',
       'cb_person_cred_hist_length_bin', 'loan_intent_bin'],
      dtype='object')

In [35]:
df_discretise["person_home_ownership"].value_counts()

person_home_ownership
RENT        15585
MORTGAGE    11791
OWN          2257
OTHER          98
Name: count, dtype: int64

In [34]:

y = df_discretise["loan_status"]


X_vars = [
    "person_home_ownership",
    "loan_intent_bin",
    "cb_person_default_on_file",
    "person_age_bin",
    "person_income_bin",
    "person_emp_length_bin",
    "loan_amnt_bin",
    "loan_percent_income_bin",
    "cb_person_cred_hist_length_bin"
]

# --- 3️⃣ Encodage en variables indicatrices (One-Hot Encoding) ---
X = pd.get_dummies(df_discretise[X_vars], drop_first=True)

# --- 4️⃣ Ajout de la constante pour l’intercept ---
X = sm.add_constant(X)

# --- 5️⃣ Conversion explicite des booléens en valeurs numériques ---
X = X.astype(float)

# --- 6️⃣ Vérification ---
print(f"Dimensions de X : {X.shape}")
print(X.dtypes.value_counts())

# --- 7️⃣ Régression logistique ---
logit_model = sm.Logit(y, X)
result = logit_model.fit(disp=1)

# --- 8️⃣ Résumé du modèle ---
print(result.summary())




Dimensions de X : (29731, 30)
float64    30
Name: count, dtype: int64
Optimization terminated successfully.
         Current function value: 0.388420
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:            loan_status   No. Observations:                29731
Model:                          Logit   Df Residuals:                    29701
Method:                           MLE   Df Model:                           29
Date:                Wed, 12 Nov 2025   Pseudo R-squ.:                  0.2680
Time:                        20:27:05   Log-Likelihood:                -11548.
converged:                       True   LL-Null:                       -15775.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                                     coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------